# Benchmarking and Improving OCR System for Southeast Asian Languages

## Data Collection

In [1]:
from article_pdf import ArticlePDF
from download import download_article_texts, download_article_pdfs, get_articles_by_language

### Wikipedia Articles

- 100 articles
- Collection of the top 20 most viewed Wikipedia articles from 5 categories ([Wikipedia:Popular pages](https://en.wikipedia.org/wiki/Wikipedia:Popular_pages))
- Format: Exact file path

In [2]:
people = ['Donald_Trump', 'Elizabeth_II', 'Barack_Obama', 'Christiano_Ronaldo', 'Michael_Jackson', 'Elon_Musk', 'Lady_Gaga', 'Adolf_Hitler', 'Eminem', 'Lionel_Messi', 'Justin_Bieber', 'Freddie_Mercury', 'Kim_Kardashian', 'Johnny_Depp', 'Steve_Jobs', 'Dwayne_Johnson', 'Michael_Jordan', 'Taylor_Swift', 'Stephen_Hawking', 'Kanye_West']
countries = ['United_States', 'India', 'United_Kingdom', 'Canada', 'Australia', 'China', 'Russia', 'Japan', 'Germany', 'France', 'Singapore', 'Israel', 'Pakistan', 'Philippines', 'Brazil', 'Italy', 'Netherlands', 'New Zealand', 'Ukraine', 'Spain']
cities = ['New_York_City', 'London', 'Hong_Kong', 'Los_Angeles', 'Dubai', 'Washington,_D.C.', 'Paris', 'Chicago', 'Angelsberg', 'Mumbai', 'San_Francisco', 'Rome', 'Monaco', 'Toronto', 'Tokyo', 'Philadelphia', 'Machu_Picchu', 'Jerusalem', 'Amsterdam', 'Boston'] # Excluding Singapore since listed for countries
life = ['Cat', 'Dog', 'Animal', 'Lion', 'Coronavirus', 'Tiger', 'Human', 'Dinosaur', 'Elephant', 'Virus', 'Horse', 'Photosynthesis', 'Evolution', 'Apple', 'Bird', 'Mammal', 'Potato', 'Polar_bear', 'Shark', 'Snake']
structures = ['Taj_Mahal', 'Burj_Khalifa', 'Statue_of_Liberty', 'Great_Wall_of_China', 'Eiffel_Tower', 'Berlin_Wall', 'Stonehenge', 'Mount_Rushmore', 'Colosseum', 'Auschwitz_concentration_camp', 'Great_Pyramid_of_Giza', 'One_World_Trade_Center', 'Empire_State_Building', 'White_House', 'Petra', 'Large_Hadron_Collider', 'Hagia_Sophia', 'Golden_Gate_Bridge', 'Panama_Canal', 'Angkor_Wat'] # Excluding Machu Picchu since listed for cities

english_titles = people + countries + cities + life + structures

english_articles = []
for english_title in english_titles:
    url = f'https://en.wikipedia.org/wiki/{english_title}'
    article = ArticlePDF(english_title, english_title, url, 'en')
    english_articles.append(article)

- To find language codes: https://www.wikidata.org/wiki/Help:Wikimedia_language_codes/lists/all
- Missing articles:
    - Thai:
        - Christiano_Ronaldo
        - Angelsberg
    - Vietnamese:
        - Christiano_Ronaldo
        - Angelsberg
    - Bahasa:
        - Christiano_Ronaldo
        - Angelsberg

In [3]:
thai_articles = get_articles_by_language(english_articles, 'th')
vietnamese_articles = get_articles_by_language(english_articles, 'vi')
bahasa_articles = get_articles_by_language(english_articles, 'id')

Issue fetching for Christiano_Ronaldo in th
Issue fetching for Angelsberg in th
Issue fetching for Christiano_Ronaldo in vi
Issue fetching for Angelsberg in vi
Issue fetching for Christiano_Ronaldo in id
Issue fetching for Angelsberg in id


In [4]:
download_article_pdfs(english_articles, 'data/english')
download_article_pdfs(thai_articles, 'data/thai')
download_article_pdfs(vietnamese_articles, 'data/vietnamese')
download_article_pdfs(bahasa_articles, 'data/bahasa')

In [4]:
download_article_texts(english_articles, 'data/english')
download_article_texts(thai_articles, 'data/thai')
download_article_texts(vietnamese_articles, 'data/vietnamese')
download_article_texts(bahasa_articles, 'data/bahasa')

## Data Proprocessing

In [2]:
from preprocess import convert_pdfs_to_pngs

convert_pdfs_to_pngs('data/english')
convert_pdfs_to_pngs('data/thai')
convert_pdfs_to_pngs('data/vietnamese')
convert_pdfs_to_pngs('data/bahasa')

## Benchmarking

### EasyOCR

In [8]:
import easyocr

en_reader = easyocr.Reader(['en']) # this needs to run only once to load the model into memory
th_reader = easyocr.Reader(['th'])
vi_reader = easyocr.Reader(['vi'])
id_reader = easyocr.Reader(['id'])

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

In [9]:
import os

def run_easy_ocr(source_path: str, reader):
    for f in os.listdir(source_path):
        print(f)
        i = 0
        image_file_path = f'{source_path}/{f}/page-{i}.png'
        res = ''
        while (os.path.exists(image_file_path)):
            text = reader.readtext(image_file_path, detail=0)
            res += ' '.join(text)
            i += 1
            image_file_path = f'{source_path}/{f}/page-{i}.png'
        with open(f'{source_path}/{f}/easy-ocr-results.txt', 'wt') as outfile:
            outfile.write(res)

In [10]:
run_easy_ocr('data/thai', th_reader)

KeyboardInterrupt: 

### Tesseract

In [11]:
import pytesseract

In [2]:
import os

def run_tesseract(source_path: str, language: str):
    for f in os.listdir(source_path):
        print(f)
        i = 0
        image_file_path = f'{source_path}/{f}/page-{i}.png'
        res = ''
        while (os.path.exists(image_file_path)):
            text = pytesseract.image_to_string(image_file_path, lang=language)
            res += ' '.join(text)
            i += 1
            image_file_path = f'{source_path}/{f}/page-{i}.png'
        with open(f'{source_path}/{f}/tesseract-results.txt', 'wt') as outfile:
            outfile.write(res)

In [15]:
run_tesseract('data/thai', 'tha')

Apple
Elon_Musk
Cat
Polar_bear
Paris
Mumbai
Mount_Rushmore
Justin_Bieber
Russia
New_York_City
Panama_Canal
Lady_Gaga
Brazil
Berlin_Wall
Eminem
Barack_Obama
Jerusalem
Stephen_Hawking
Ukraine
Washington,_D.C.
France
Great_Pyramid_of_Giza
Colosseum
Machu_Picchu
Dog
Netherlands
Auschwitz_concentration_camp
Michael_Jackson
Chicago
Potato
Spain
Kanye_West
Photosynthesis
Tokyo
Australia
Mammal


KeyboardInterrupt: 

In [24]:
source_path = 'data/thai'
i = 0
for f in os.listdir(source_path):
    file_path = f'{source_path}/{f}/tesseract-results.txt'
    if os.path.exists(file_path):
        i += 1
print(i)

# EasyOCR: 30 in 40 minutes
# Tesseract: 35 in 20 minutes

35


In [3]:
from torchmetrics.text import CharErrorRate
cer = CharErrorRate()

# predictions = ["this is the prediction", "there is an other sample"]
# targets = ["this is the reference", "there is another one"]
# print(cer(predictions, targets))

# # 0.0 CER
# predictions = ["100% match", "this is some text"]
# targets = ["100% match", "this is some text"]
# print(cer(predictions, targets))

source_path = 'data/thai'

predictions = []
targets = []
for f in os.listdir(source_path):
    prediction_file_path = f'{source_path}/{f}/tesseract-results.txt'
    target_file_path = f'{source_path}/{f}/text.txt'
    try:
        prediction_file = open(prediction_file_path, "r")
        target_file = open(target_file_path, "r")
        predictions.append(prediction_file.read())
        targets.append(target_file.read())
    except:
        print('File does not exist')
        
test_prediction = [predictions[0]]
test_target = [targets[0]]
print('yay')
print(cer(test_prediction, test_target))

File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
File does not exist
